In [2]:
import importlib
from datetime import datetime, timezone

# Reload local package modules so this notebook picks up provider changes
# without requiring a full kernel restart.
for module_name in (
    "investment_adviser.config",
    "investment_adviser.providers.symbols",
    "investment_adviser.providers.libertex",
    "investment_adviser.providers.fallback",
    "investment_adviser",
):
    module = importlib.import_module(module_name)
    importlib.reload(module)

from investment_adviser import (
    find_libertex_instruments,
    load_symbol_data,
    perform_technical_analysis,
    perform_candle_analysis,
    perform_symbol_sentiment_analysis,
)
from investment_adviser.exceptions import DataProviderError


In [3]:
tickers = find_libertex_instruments()
len(tickers), tickers[:20]


(275,
 ['ADAUSD',
  'Adidas',
  'Adobe',
  'AF',
  'AGG',
  'AIR',
  'Alcoa',
  'Alibaba',
  'Amazon',
  'AMC',
  'American_Express',
  'AMX',
  'APEUSD',
  'Apple',
  'AT&T',
  'ATMUSD',
  'AUDCAD',
  'AUDCHF',
  'AUDJPY',
  'AUDNZD'])

In [4]:
with open("tickers.txt", "w", encoding="utf-8") as file_obj:
    for ticker in tickers:
        file_obj.write(f"{ticker}\n")


In [ ]:
begin_time = datetime(2024, 1, 1, tzinfo=timezone.utc)
end_time = datetime.now(timezone.utc)

clear_tickers = []
failed_tickers = []

for index, ticker in enumerate(tickers, start=1):
    try:
        data = load_symbol_data(
            symbol=ticker,
            timeframe="1d",
            begin_time=begin_time,
            end_time=end_time,
            provider="fallback",
        )
        clear_tickers.append(ticker)
    except Exception as exc:
        failed_tickers.append((ticker, f"{type(exc).__name__}: {exc}"))

    if index % 25 == 0:
        print(
            f"checked={index} "
            f"clear={len(clear_tickers)} "
            f"failed={len(failed_tickers)}"
        )

with open("clear_tickers.txt", "w", encoding="utf-8") as file_obj:
    for ticker in clear_tickers:
        file_obj.write(f"{ticker}\n")

with open("failed_tickers.txt", "w", encoding="utf-8") as file_obj:
    for ticker, reason in failed_tickers:
        file_obj.write(f"{ticker}: {reason}\n")


In [ ]:
{
    "tickers": len(tickers),
    "clear_tickers": len(clear_tickers),
    "failed_tickers": len(failed_tickers),
    "first_failures": failed_tickers[:10],
}


{'tickers': 281,
 'clear_tickers': 256,
 'failed_tickers': 25,
 'first_failures': [('AT',
   'DataProviderError: No fallback market data provider succeeded. yfinance: No yfinance market data returned for AT. Tried: AT: empty response | binance: No Binance market data returned for AT. Tried: no crypto aliases | stooq: No Stooq market data returned for AT. Tried: at.us: empty response, at: empty response'),
  ('ATVI',
   'DataProviderError: No fallback market data provider succeeded. yfinance: No yfinance market data returned for ATVI. Tried: ATVI: empty response | binance: No Binance market data returned for ATVI. Tried: no crypto aliases | stooq: No Stooq market data returned for ATVI. Tried: atvi.us: empty response, atvi: empty response'),
  ('BRETTUSD',
   'DataProviderError: No fallback market data provider succeeded. yfinance: No yfinance market data returned for BRETTUSD. Tried: BRETT-USD: empty response, BRETTUSD: empty response | binance: No Binance market data returned for BRET